In [1]:
import pandas as pd

In [4]:
train_df = pd.read_csv("data/lemmantized/train.csv")

In [2]:
#df[df['link']=='https://ceticismopolitico.com/2017/11/30/katia-abreu-diz-que-vai-colocar-sua-expulsao-em-uma-moldura-mas-nao-para-de-reclamar/']

In [22]:
from utils import clean_text, SpacyPreprocessor

import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

def compare_preprocessed_models(lemma_path, simple_path):
    # Load Datasets
    df_lemma = pd.read_parquet(lemma_path)
    df_simple = pd.read_parquet(simple_path)
    
    # Define a standard model configuration
    # Using float32 saves 50% RAM in WSL
    def get_pipe():
        return Pipeline([
            ("tfidf", TfidfVectorizer(max_features=5000, ngram_range=(1,2), dtype=np.float32)),
            ("clf", LogisticRegression(n_jobs=-1, max_iter=1000))
        ])

    datasets = [
        ("Lemmatized", df_lemma["text"], df_lemma["label"]),
        ("Simple Clean", df_simple["text"], df_simple["label"])
    ]

    for name, X, y in datasets:
        print(f"\nEvaluating {name}...")
        # CV is better than a single split for your research
        scores = cross_val_score(get_pipe(), X, y, cv=5, scoring='f1')
        print(f"{name} F1-Score: {scores.mean():.4f} (+/- {scores.std() * 2:.2f})")

# Usage
compare_preprocessed_models("train_lemmatized.parquet", "train_simple_cleaned.parquet")


Evaluating Lemmatized...


/home/victor/training/uninter/NLP/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
/home/victor/training/uninter/NLP/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
/home/victor/training/uninter/NLP/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
/home/victor/training/uninter/NLP/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1184: FutureWarning: 'n_jobs' has no ef

Lemmatized F1-Score: 0.9528 (+/- 0.01)

Evaluating Simple Clean...


/home/victor/training/uninter/NLP/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
/home/victor/training/uninter/NLP/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
/home/victor/training/uninter/NLP/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
/home/victor/training/uninter/NLP/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1184: FutureWarning: 'n_jobs' has no ef

Simple Clean F1-Score: 0.9528 (+/- 0.01)


In [14]:
train_df["text"].str.len().describe()

count     5040.000000
mean      2859.345040
std       2994.646281
min         44.000000
25%        693.750000
50%       1598.500000
75%       4129.250000
max      34009.000000
Name: text, dtype: float64

In [15]:
print("Train max length:", train_df["text"].str.len().max())
print("Train count:", train_df.shape[0])

Train max length: 34009
Train count: 5040


'content based model'

* clean texts
* truncate texts 


In [9]:
import pandas as pd
import numpy as np
import os
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report

# --------------------------------------------------
# RESOURCE & DATA SETUP
# --------------------------------------------------
# SVC RBF is single-threaded in scikit-learn's fit, 
# so we use n_jobs in GridSearchCV instead.
total_cores = os.cpu_count()
safe_cores = max(1, total_cores - 1)

# Load the "winner" dataset (e.g., the lemmatized one)
name = "lemma" # Change to your best performing folder
train_df = pd.read_csv(f"./data/lemmantized/train.csv").dropna(subset=['text', 'label'])

# RBF is O(n_samples^2). If dataset > 10k, WSL might crash.
# We take a representative sample for the optimization phase.
if len(train_df) > 10000:
    train_df = train_df.sample(10000, random_state=42)

X_train, y_train = train_df['text'], train_df["label"]

# --------------------------------------------------
# GRID SEARCH SETUP
# --------------------------------------------------
pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(max_features=5000, dtype=np.float32)),
    ("clf", SVC(kernel='rbf'))
])

# Range of C and Gamma to test
param_grid = {
    'clf__C': [0.1, 1, 10, 100],
}

print(f"🚀 Starting RBF Optimization on {len(X_train)} samples...")

grid_search = GridSearchCV(
    pipeline, 
    param_grid, 
    cv=2, 
    scoring='f1', 
    n_jobs=safe_cores, 
    verbose=2
)

grid_search.fit(X_train, y_train)

# --------------------------------------------------
# SAVE RESULTS
# --------------------------------------------------
results_df = pd.DataFrame(grid_search.cv_results_)
os.makedirs('./results', exist_ok=True)
results_df.to_csv("./results/svc_rbf_optimization.csv", index=False)

print("\n🏆 Best Parameters:", grid_search.best_params_)
print(f"🏆 Best F1-Score: {grid_search.best_score_:.4f}")

🚀 Starting RBF Optimization on 5760 samples...
Fitting 2 folds for each of 4 candidates, totalling 8 fits
[CV] END ...........................................clf__C=1; total time=  16.1s
[CV] END .........................................clf__C=0.1; total time=  21.6s
[CV] END .........................................clf__C=0.1; total time=  21.6s
[CV] END ...........................................clf__C=1; total time=  17.2s
[CV] END ..........................................clf__C=10; total time=  16.8s
[CV] END ..........................................clf__C=10; total time=  17.2s
[CV] END .........................................clf__C=100; total time=  16.9s
[CV] END .........................................clf__C=100; total time=  18.4s

🏆 Best Parameters: {'clf__C': 10}
🏆 Best F1-Score: 0.9541


In [12]:
import pandas as pd
import numpy as np
import os
import optuna
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC
from sklearn.model_selection import cross_val_score

# --------------------------------------------------
# DATA SETUP
# --------------------------------------------------
name = "lemmantized" 
train_df = pd.read_csv(f"./data/{name}/train.csv").dropna(subset=['text', 'label'])

# Scale down for RBF performance in WSL
if len(train_df) > 5000:
    train_df = train_df.sample(5000, random_state=42)

X_train, y_train = train_df['text'], train_df["label"]

# --------------------------------------------------
# OPTUNA OBJECTIVE FUNCTION
# --------------------------------------------------
def objective(trial):
    # Suggest a value for C on a logarithmic scale
    # This is more effective for C than a linear range
    c_param = trial.suggest_float('C', 0.1, 100.0, log=True)
    gamma_param = trial.suggest_categorical('gamma', ['scale', 'auto', 0.1, 0.01])
    
    pipeline = Pipeline([
        ("tfidf", TfidfVectorizer(max_features=5000, dtype=np.float32)),
        ("clf", SVC(kernel='rbf', C=c_param, gamma=gamma_param))
    ])

    # Using 3-fold CV to evaluate the suggestion
    score = cross_val_score(pipeline, X_train, y_train, n_jobs=-1, cv=3, scoring='f1')
    return score.mean()

# --------------------------------------------------
# RUN STUDY
# --------------------------------------------------
# 'maximize' because we want the highest F1-score
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=30) # 30 trials is a good start for one parameter

print("\n🏆 Best trial:")
trial = study.best_trial
print(f"  F1-Score: {trial.value:.4f}")
print(f"  Best C: {trial.params['C']}")

# Save results to CSV
results_df = study.trials_dataframe()
os.makedirs('./results', exist_ok=True)
results_df.to_csv("./results/optuna_svc_results.csv", index=False)

/home/victor/training/uninter/NLP/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[I 2026-03-09 18:20:07,799] A new study created in memory with name: no-name-915d384d-aca1-48c9-a55e-9d1586bf1949
[I 2026-03-09 18:20:17,402] Trial 0 finished with value: 0.955322978226746 and parameters: {'C': 82.95475127409898, 'gamma': 0.1}. Best is trial 0 with value: 0.955322978226746.
[I 2026-03-09 18:20:28,809] Trial 1 finished with value: 0.9563963426316769 and parameters: {'C': 42.37949510255788, 'gamma': 0.01}. Best is trial 1 with value: 0.9563963426316769.
[I 2026-03-09 18:20:42,806] Trial 2 finished with value: 0.9507766127599075 and parameters: {'C': 1.534949902757033, 'gamma': 0.1}. Best is trial 1 with value: 0.9563963426316769.
[I 2026-03-09 18:21:01,325] Trial 3 finished with value: 0.93389063624407


🏆 Best trial:
  F1-Score: 0.9584
  Best C: 8.172787520025306


In [ ]:
import pandas as pd
import numpy as np
import os
import optuna
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC
from sklearn.model_selection import cross_val_score

# --------------------------------------------------
# DATA SETUP
# --------------------------------------------------
name = "lemmantized" 
train_df = pd.read_csv(f"./data/{name}/train.csv").dropna(subset=['text', 'label'])

# Scale down for RBF performance in WSL
if len(train_df) > 5000:
    train_df = train_df.sample(5000, random_state=42)

X_train, y_train = train_df['text'], train_df["label"]

# --------------------------------------------------
# OPTUNA OBJECTIVE FUNCTION
# --------------------------------------------------
def objective(trial):
    # Suggest a value for C on a logarithmic scale
    # This is more effective for C than a linear range
    c_param = trial.suggest_float('C', 0.1, 100.0, log=True)
    gamma_param = trial.suggest_categorical('gamma', ['scale', 'auto', 0.1, 0.01])
    
    pipeline = Pipeline([
        ("tfidf", TfidfVectorizer(max_features=5000, dtype=np.float32)),
        ("clf", SVC(kernel='rbf', C=c_param, gamma=gamma_param))
    ])

    # Using 3-fold CV to evaluate the suggestion
    score = cross_val_score(pipeline, X_train, y_train, n_jobs=-1, cv=3, scoring='f1')
    return score.mean()

# --------------------------------------------------
# RUN STUDY
# --------------------------------------------------
# 'maximize' because we want the highest F1-score
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=30) # 30 trials is a good start for one parameter

print("\n🏆 Best trial:")
trial = study.best_trial
print(f"  F1-Score: {trial.value:.4f}")
print(f"  Best C: {trial.params['C']}")

# Save results to CSV
results_df = study.trials_dataframe()
os.makedirs('./results', exist_ok=True)
results_df.to_csv("./results/optuna_svc_results.csv", index=False)

In [14]:
trial

FrozenTrial(number=19, state=<TrialState.COMPLETE: 1>, values=[0.9583733988283498], datetime_start=datetime.datetime(2026, 3, 9, 18, 25, 4, 661108), datetime_complete=datetime.datetime(2026, 3, 9, 18, 25, 13, 463589), params={'C': 8.172787520025306, 'gamma': 0.1}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'C': FloatDistribution(high=100.0, log=True, low=0.1, step=None), 'gamma': CategoricalDistribution(choices=('scale', 'auto', 0.1, 0.01))}, trial_id=19, value=None)